<a href="https://colab.research.google.com/github/rpaivadias69/MALE_Abgaben_PVA-s_Nachbearbeitungen/blob/PVA1_Raulpaivadias_Nachbearbeitung1/MALE_PVA1_Nachbereitungsaufgaben_RaulPaivaDias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Starting with exercises for chapter 1:

1: Machine Learning beschäftigt sich damit, Programme zu entwickeln, die eigenständig aus Daten Muster erkennen. Ein System „lernt“, wenn es mit der Zeit bessere Ergebnisse bei einer bestimmten Aufgabe erzielt – gemessen an einem definierten Leistungsmaß.


2: Machine Learning ist besonders sinnvoll:

bei sehr komplexen Problemen, für die man keine festen Regeln programmieren kann,

wenn man starre, manuell erstellte Regelwerke ersetzen möchte,

wenn sich die Umgebung oder Daten häufig ändern,

und wenn große Datenmengen analysiert werden sollen, um neue Zusammenhänge zu entdecken.

3: Ein gelabelter Trainingsdatensatz enthält für jedes Beispiel die korrekte Zielausgabe. Diese vorgegebenen Lösungen dienen dem Modell als Orientierung beim Lernen.

4: Die beiden wichtigsten Arten von überwachten Lernaufgaben sind:

Klassifikation, bei der Daten in Kategorien eingeteilt werden,

Regression, bei der ein konkreter Zahlenwert vorhergesagt wird.

5: Beim unüberwachten Lernen geht es häufig um:

das Bilden von Gruppen ähnlicher Datenpunkte (Clustering),

das Vereinfachen von Daten durch Reduktion der Dimension,

die visuelle Darstellung komplexer Daten,

oder das Erkennen typischer Zusammenhänge zwischen Variablen.

6: Soll ein Roboter lernen, sich in unbekanntem Gelände fortzubewegen, bietet sich Reinforcement Learning an. Dabei probiert das System verschiedene Handlungen aus und erhält Rückmeldungen in Form von Belohnungen oder Strafen.

7: Wenn noch nicht feststeht, welche Kundengruppen existieren, kann man mithilfe von Clustering automatisch ähnliche Kunden zusammenfassen. Sind die gewünschten Gruppen jedoch bereits definiert, kann ein Klassifikationsmodell trainiert werden, das neue Kunden entsprechend einordnet.

8: Die Erkennung von Spam wird normalerweise als überwachte Lernaufgabe behandelt, da das Modell mit E-Mails trainiert wird, die bereits als „Spam“ oder „Nicht-Spam“ markiert sind.

9: Ein Online-Lernsystem wird fortlaufend mit neuen Daten aktualisiert. Im Gegensatz zum Batch-Lernen erfolgt das Training nicht nur einmalig, sondern schrittweise.

10: Out-of-core-Lernen wird eingesetzt, wenn die Datenmenge zu groß für den Hauptspeicher ist. In diesem Fall werden kleinere Datenblöcke nacheinander verarbeitet.

11: Instanzbasierte Verfahren speichern Trainingsbeispiele und vergleichen neue Daten mit diesen. Die Vorhersage basiert darauf, welche gespeicherten Beispiele am ähnlichsten sind.

12: Modellparameter beeinflussen direkt die Vorhersagen eines Modells. Hyperparameter hingegen steuern den Lernprozess selbst und werden vor dem Training festgelegt.

13: Modellbasierte Verfahren versuchen, passende Parameterwerte zu finden, sodass das Modell auch auf unbekannten Daten gute Ergebnisse liefert. Dazu wird meist eine Fehlerfunktion minimiert, die misst, wie stark die Vorhersagen von den tatsächlichen Werten abweichen.

14: Typische Probleme im Machine Learning sind:

zu wenig oder ungeeignete Daten,

fehlerhafte oder verzerrte Datensätze,

ungeeignete Merkmalsauswahl,

zu einfache Modelle (Underfitting),

zu komplexe Modelle (Overfitting).

15: Wenn ein Modell auf Trainingsdaten sehr gut abschneidet, bei neuen Daten aber schlecht funktioniert, spricht man von Overfitting. Abhilfe schaffen kann man zum Beispiel durch mehr Trainingsdaten, ein einfacheres Modell oder stärkere Regularisierung.

16: Der Testdatensatz dient dazu, die tatsächliche Leistungsfähigkeit eines Modells auf unbekannten Daten zu überprüfen.

17: Ein Validierungsdatensatz hilft dabei, verschiedene Modelle zu vergleichen und optimale Einstellungen (Hyperparameter) auszuwählen.

18: Ein Train-Dev-Set wird eingesetzt, wenn sich Trainingsdaten und spätere Anwendungsdaten möglicherweise unterscheiden. Es ermöglicht zu erkennen, ob ein Problem durch Overfitting oder durch unterschiedliche Datenverteilungen entsteht.

19: Werden Hyperparameter mithilfe des Testdatensatzes angepasst, kann das Modell unbewusst auf diesen speziellen Datensatz optimiert werden. Das führt zu unrealistisch guten Testergebnissen und einer verzerrten Einschätzung der tatsächlichen Leistung.

In [ ]:
# Following the exercises to chapter 2
# A comment on on what each code section does can be found before the start of each section
# It starts with the code needed, to execute the exercises

# ============================================
# 0) Imports

import numpy as np
import pandas as pd
from pathlib import Path
import tarfile
import urllib.request

from sklearn.model_selection import StratifiedShuffleSplit, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans
from sklearn.svm import SVR


# ============================================
# 1) Load dataset

def fetch_housing_csv():
    url = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.tgz"
    data_dir = Path("datasets/housing")
    data_dir.mkdir(parents=True, exist_ok=True)
    tgz_path = data_dir / "housing.tgz"

    if not tgz_path.exists():
        urllib.request.urlretrieve(url, tgz_path)
        with tarfile.open(tgz_path) as tgz:
            tgz.extractall(path=data_dir)

    csv_path = data_dir / "housing.csv"
    return pd.read_csv(csv_path)

data_full = fetch_housing_csv()


# ============================================
# 2) Stratified split

data_full["income_cat"] = pd.cut(
    data_full["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in splitter.split(data_full, data_full["income_cat"]):
    train_set = data_full.loc[train_idx].drop("income_cat", axis=1)
    test_set  = data_full.loc[test_idx].drop("income_cat", axis=1)

# separate predictors + labels
train_y = train_set["median_house_value"].copy()
train_X = train_set.drop("median_house_value", axis=1)

test_y = test_set["median_house_value"].copy()
test_X = test_set.drop("median_house_value", axis=1)


# ============================================
# 3) Preprocessing

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(
            n_clusters=self.n_clusters,
            n_init=10,
            random_state=self.random_state
        )
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"cluster_{i}_similarity" for i in range(self.n_clusters)]


def _column_ratio(arr):
    return arr[:, [0]] / arr[:, [1]]

def _ratio_feature_name(transformer, feature_names_in):
    return ["ratio"]

def make_ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(_column_ratio, feature_names_out=_ratio_feature_name),
        StandardScaler()
    )

log_transform_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler()
)

default_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

geo_cluster_pipeline = ClusterSimilarity(n_clusters=10, gamma=1.0, random_state=42)

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

# Columns
numeric_cols = [
    "longitude", "latitude", "housing_median_age", "total_rooms",
    "total_bedrooms", "population", "households", "median_income"
]
categorical_cols = ["ocean_proximity"]

# Build the full preprocessing transformer
prep_pipe = ColumnTransformer([
    ("bedrooms_per_rooms", make_ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_household", make_ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_household", make_ratio_pipeline(), ["population", "households"]),
    ("log_features", log_transform_pipeline, ["total_bedrooms", "total_rooms", "population", "households", "median_income"]),
    ("geo_clusters", geo_cluster_pipeline, ["latitude", "longitude"]),
    ("num", default_num_pipeline, numeric_cols),
    ("cat", cat_pipeline, categorical_cols),
])




GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('prep',
                                        ColumnTransformer(transformers=[('bedrooms_per_rooms',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('functiontransformer',
                                                                                          FunctionTransformer(feature_names_out=<function _ratio_feature_name at 0x7a40eea93240>,
                                                                                                              func=<function _column_ratio at 0x7a40eea93740>)),
                                                                                         ('standardscaler',
                                                                                          Stand...
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehotencoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['ocean_proximity'])])),
                                       ('svr', SVR())]),
             param_grid=[{'svr__C': [10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0,
                                     10000.0],
                          'svr__kernel': ['linear']},
                         {'svr__C': [1.0, 3.0, 10.0, 30.0, 100.0, 300.0,
                                     1000.0],
                          'svr__gamma': [0.01, 0.03, 0.1, 0.3, 1.0],
                          'svr__kernel': ['rbf']}],
             scoring='neg_root_mean_squared_error')

In [ ]:
# ============================================
# EXERCISE 1: SVR with GridSearchCV

svr_modeling = Pipeline([
    ("prep", prep_pipe),
    ("svr", SVR())
])

svr_param_grid = [
    {
        "svr__kernel": ["linear"],
        "svr__C": [10., 30., 100., 300., 1000., 3000., 10000.]
    },
    {
        "svr__kernel": ["rbf"],
        "svr__C": [1., 3., 10., 30., 100., 300., 1000.],
        "svr__gamma": [0.01, 0.03, 0.1, 0.3, 1.0]
    }
]

svr_grid = GridSearchCV(
    svr_modeling,
    svr_param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error"
)

display(svr_grid)

# only first 5000 instances
svr_grid.fit(train_X.iloc[:5000], train_y.iloc[:5000])

best_rmse = -svr_grid.best_score_
print("Best CV RMSE (SVR):", best_rmse)
print("Best parameters:", svr_grid.best_params_)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('prep',
                                        ColumnTransformer(transformers=[('bedrooms_per_rooms',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('functiontransformer',
                                                                                          FunctionTransformer(feature_names_out=<function _ratio_feature_name at 0x7a40eea93240>,
                                                                                                              func=<function _column_ratio at 0x7a40eea93740>)),
                                                                                         ('standardscaler',
                                                                                          Stand...
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehotencoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['ocean_proximity'])])),
                                       ('svr', SVR())]),
             param_grid=[{'svr__C': [10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0,
                                     10000.0],
                          'svr__kernel': ['linear']},
                         {'svr__C': [1.0, 3.0, 10.0, 30.0, 100.0, 300.0,
                                     1000.0],
                          'svr__gamma': [0.01, 0.03, 0.1, 0.3, 1.0],
                          'svr__kernel': ['rbf']}],
             scoring='neg_root_mean_squared_error')

Best CV RMSE (SVR): 64696.98669819072
Best parameters: {'svr__C': 10000.0, 'svr__kernel': 'linear'}


In [ ]:
# EXERCISE 2: RandomizedSearchCV

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

# ============================================
# 1) but build a fresh SVR pipeline object

svr_flow = Pipeline([
    ("prep", prep_pipe),
    ("model", SVR())
])

# ============================================
# 2) Random search space:
search_space = {
    "model__kernel": ["linear", "rbf"],
    "model__C": loguniform(1e1, 3e4),
    "model__gamma": loguniform(1e-3, 1.0)
}

rand_search = RandomizedSearchCV(
    estimator=svr_flow,
    param_distributions=search_space,
    n_iter=30,  #If we want a longer search, we would increase the n_iter
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1
)

# ============================================
# 3) first 5000 rows
rand_search.fit(train_X.iloc[:5000], train_y.iloc[:5000])

best_rmse_rand = -rand_search.best_score_
print("RandomizedSearch best CV RMSE:", best_rmse_rand)
print("Best sampled parameters:", rand_search.best_params_)


rand_search

RandomizedSearch best CV RMSE: 62874.27512544064
Best sampled parameters: {'model__C': np.float64(23515.99274057193), 'model__gamma': np.float64(0.2115429079726121), 'model__kernel': 'rbf'}


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('prep',
                                              ColumnTransformer(transformers=[('bedrooms_per_rooms',
                                                                               Pipeline(steps=[('simpleimputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('functiontransformer',
                                                                                                FunctionTransformer(feature_names_out=<function _ratio_feature_name at 0x7a40eea93240>,
                                                                                                                    func=<function _column_ratio at 0x7a40eea93740>)),
                                                                                               ('standardscaler'...
                                                                               ['ocean_proximity'])])),
                                             ('model', SVR())]),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'model__C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7a40f2a6ea20>,
                                        'model__gamma': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7a40eebfc320>,
                                        'model__kernel': ['linear', 'rbf']},
                   random_state=42, scoring='neg_root_mean_squared_error')

In [ ]:
# EXERCISE 3: SelectFromModel

from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
import numpy as np
import pandas as pd

# ============================================
# 1) Take the best SVR hyperparameters found in Exercise 2
chosen_kernel = rand_search.best_params_["model__kernel"]
chosen_C = rand_search.best_params_["model__C"]


chosen_gamma = rand_search.best_params_.get("model__gamma", "scale")

# ============================================
# 2) Build a pipeline: preprocessing -> feature selection -> SVR

rf_selector = SelectFromModel(
    estimator=RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    threshold=0.005
)

svr_after_selection = Pipeline([
    ("prep", prep_pipe),
    ("select", rf_selector),
    ("svr", SVR(kernel=chosen_kernel, C=chosen_C, gamma=chosen_gamma))
])

# ============================================
# 3) Evaluate with 3-fold CV on first 5000 rows (RMSE)
neg_rmse_scores = cross_val_score(
    svr_after_selection,
    train_X.iloc[:5000],
    train_y.iloc[:5000],
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1
)

rmse_scores = -neg_rmse_scores
print("RMSE per fold:", rmse_scores)
print("\nSummary:")
display(pd.Series(rmse_scores).describe())

RMSE per fold: [59428.91593486 60855.66047404 58641.61388348]

Summary:


,0
count,3.000000
mean,59642.063431
std,1122.307653
min,58641.613883
25%,59035.264909
50%,59428.915935
75%,60142.288204
max,60855.660474


In [ ]:
# EXERCISE 4: Custom Transformer, Integration into preprocessing
# Preparation-Cell

# ============================================
# 1) Creating custom transformer

from sklearn.base import BaseEstimator, TransformerMixin, MetaEstimatorMixin, clone
from sklearn.utils.validation import check_array, check_is_fitted
import numpy as np

class PredictedValueFeature(MetaEstimatorMixin, TransformerMixin, BaseEstimator):

    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y=None):
        X = check_array(X)
        self._model = clone(self.estimator)
        self._model.fit(X, y)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self, "_model")
        X = check_array(X)
        preds = self._model.predict(X)
        if preds.ndim == 1:
            preds = preds.reshape(-1, 1)
        return preds

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "_model")
        name = self._model.__class__.__name__.lower()
        return np.array([f"{name}_price_hint"])

# ============================================
# 3) Verifying

from sklearn.neighbors import KNeighborsRegressor

geo_X = train_X[["latitude", "longitude"]].iloc[:5000]
geo_y = train_y.iloc[:5000]

knn_hint = PredictedValueFeature(
    KNeighborsRegressor(n_neighbors=3, weights="distance")
)

knn_hint.fit(geo_X, geo_y)
knn_hint.transform(geo_X[:5]), knn_hint.get_feature_names_out()


# ============================================
# 2) Replacing old geo transformer

from sklearn.compose import ColumnTransformer

# Build a list of the existing transformers, but swap out the geo part
updated_blocks = []
for name, trans, cols in prep_pipe.transformers:
    if name == "geo_clusters":
        updated_blocks.append(("geo_knn_hint", knn_hint, ["latitude", "longitude"]))
    else:
        updated_blocks.append((name, trans, cols))

prep_with_knn_geo = ColumnTransformer(updated_blocks)

In [ ]:
# Solution-Cell

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVR
import pandas as pd

# Take best SVR params from exercise 2
k = rand_search.best_params_["model__kernel"]
c = rand_search.best_params_["model__C"]
g = rand_search.best_params_.get("model__gamma", "scale")

svr_with_knn_geo = Pipeline([
    ("prep", prep_with_knn_geo),
    ("svr", SVR(kernel=k, C=c, gamma=g))
])

scores = cross_val_score(
    svr_with_knn_geo,
    train_X.iloc[:5000],
    train_y.iloc[:5000],
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rmse_vals = -scores
print("Fold RMSEs:", rmse_vals)
display(pd.Series(rmse_vals).describe())

Fold RMSEs: [111410.67624141 114604.02251567 106918.7241454 ]


,0
count,3.000000
mean,110977.807634
std,3860.891631
min,106918.724145
25%,109164.700193
50%,111410.676241
75%,113007.349379
max,114604.022516


In [ ]:
# EXERCISE 5: RandomSearchCV

# ============================================
# 1) Imports

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, loguniform
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR

# ============================================
# 2) Pipeline -> SVR
auto_tune_pipe = Pipeline([
    ("prep", prep_with_knn_geo),
    ("svr", SVR(kernel="rbf"))
])

# ============================================
# 3) tuning BOTH: geo-KNN settings & SVR hyperparameters

param_space_5 = {
    # KNN settings
    "prep__geo_knn_hint__estimator__n_neighbors": randint(1, 30),
    "prep__geo_knn_hint__estimator__weights": ["uniform", "distance"],

    # SVR settings
    "svr__C": loguniform(20, 200_000),
    "svr__gamma": loguniform(1e-3, 1.0),
}

prep_rand_search = RandomizedSearchCV(
    estimator=auto_tune_pipe,
    param_distributions=param_space_5,
    n_iter=60,   # Above used 30, so using 60 now
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1
)

prep_rand_search.fit(train_X.iloc[:5000], train_y.iloc[:5000])

best_rmse_5 = -prep_rand_search.best_score_
print("Exercise 5 best CV RMSE:", best_rmse_5)
print("Best config:", prep_rand_search.best_params_)

prep_rand_search

Exercise 5 best CV RMSE: 98280.74646034524
Best config: {'prep__geo_knn_hint__estimator__n_neighbors': 1, 'prep__geo_knn_hint__estimator__weights': 'uniform', 'svr__C': np.float64(10564.569436244974), 'svr__gamma': np.float64(0.039156489582430024)}


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('prep',
                                              ColumnTransformer(transformers=[('bedrooms_per_rooms',
                                                                               Pipeline(steps=[('simpleimputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('functiontransformer',
                                                                                                FunctionTransformer(feature_names_out=<function _ratio_feature_name at 0x7a40eea93240>,
                                                                                                                    func=<function _column_ratio at 0x7a40eea93740>)),
                                                                                               ('standardscaler'...
                   param_distributions={'prep__geo_knn_hint__estimator__n_neighbors': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7a40eeb62cc0>,
                                        'prep__geo_knn_hint__estimator__weights': ['uniform',
                                                                                   'distance'],
                                        'svr__C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7a40ec6c2f00>,
                                        'svr__gamma': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7a40eec14110>},
                   random_state=42, scoring='neg_root_mean_squared_error')

In [ ]:
#EXERCISE 6: STandardScalerClone

# ============================================
# 1) Preparation

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted, validate_data


class ZScoreScalerClone(TransformerMixin, BaseEstimator):

    #Minimal clone of StandardScaler:

    def __init__(self, with_mean=True):
        self.with_mean = with_mean

    def fit(self, X, y=None):
        # store feature names if DataFrame
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)

        Xv = validate_data(self, X, ensure_2d=True)
        self.n_features_in_ = Xv.shape[1]

        if self.with_mean:
            self.mean_ = Xv.mean(axis=0)
        else:
            self.mean_ = None

        self.scale_ = Xv.std(axis=0, ddof=0)
        # avoid division by zero
        self.scale_[self.scale_ == 0] = 1.0
        return self

    def transform(self, X):
        check_is_fitted(self, "scale_")
        Xv = validate_data(self, X, ensure_2d=True, reset=False)

        Z = Xv / self.scale_
        if self.with_mean:
            Z = (Xv - self.mean_) / self.scale_
        return Z

    def inverse_transform(self, X):
        check_is_fitted(self, "scale_")
        Xv = validate_data(self, X, ensure_2d=True, reset=False)

        X_back = Xv * self.scale_
        if self.with_mean:
            X_back = X_back + self.mean_
        return X_back

    def get_feature_names_out(self, input_features=None):

        # implementing Rules from the exercise

        check_is_fitted(self, "scale_")

        if input_features is None:
            if hasattr(self, "feature_names_in_"):
                return self.feature_names_in_
            return np.asarray([f"x{i}" for i in range(self.n_features_in_)], dtype=object)

        input_features = np.asarray(input_features, dtype=object)

        if len(input_features) != self.n_features_in_:
            raise ValueError(
                f"input_features has length {len(input_features)} but expected {self.n_features_in_}"
            )

        if hasattr(self, "feature_names_in_"):
            if not np.array_equal(input_features, self.feature_names_in_):
                raise ValueError("input_features does not match feature_names_in_")

        return input_features

In [ ]:
# ============================================
# 2) Estimator API
# Idea taken from the Book's Notebook

from sklearn.utils.estimator_checks import check_estimator

check_estimator(ZScoreScalerClone())
print("Estimator checks passed ✅")

Estimator checks passed ✅


In [ ]:
# ============================================
# 3) Functional Tests, also according and with help from the book (And a bit of ChatGPT)

# basic behavior
np.random.seed(42)
M = np.random.rand(1000, 3)

sc = ZScoreScalerClone()
M_scaled = sc.fit_transform(M)

assert np.allclose(M_scaled, (M - M.mean(axis=0)) / M.std(axis=0))

# with_mean=False behavior
sc2 = ZScoreScalerClone(with_mean=False)
M_scaled2 = sc2.fit_transform(M)

assert np.allclose(M_scaled2, M / M.std(axis=0))

# inverse_transform should undo transform
sc3 = ZScoreScalerClone()
M_back = sc3.inverse_transform(sc3.fit_transform(M))

assert np.allclose(M, M_back)

# feature names out (no DataFrame)
assert np.all(sc3.get_feature_names_out() == np.array(["x0", "x1", "x2"], dtype=object))
assert np.all(sc3.get_feature_names_out(["a", "b", "c"]) == np.array(["a", "b", "c"], dtype=object))

# DataFrame feature_names_in_ handling
df = pd.DataFrame({"a": np.random.rand(100), "b": np.random.rand(100)})
sc4 = ZScoreScalerClone()
_ = sc4.fit_transform(df)

assert np.all(sc4.feature_names_in_ == np.array(["a", "b"], dtype=object))
assert np.all(sc4.get_feature_names_out() == np.array(["a", "b"], dtype=object))

print("All manual tests passed ✅")

All manual tests passed ✅
